In [1]:
import numpy as np
import sqlite3
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN
from lmfit.models import GaussianModel
from functools import reduce
from lmfit import Parameter
from astropy.constants import c
from astropy.units import Quantity
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import re


h = 6.626*10**(-34)  # Planck's constant in J*s
h_csg = 6.62607015*10**(-27)  # Planck's constant in erg*s
k = 1.381*10**(-23)  # Boltzmann's constant in J/K
k_B = 1.380649*10**(-16)  # Boltzmann's constant in erg/K 
kcm = 0.69503476  # Boltzmann's constant in cm-1/K
ckm = 2.998*10**5  # peed of light in km/s
ccm = 2.998*10**10  # speed of light in cm/s
cm = 2.998*10**8  # speed of light in m/s

In [ ]:
def gauss_fit(name, freq_l, freq_u, step, db_path, p, beam=None, Save=False, save_file=None):


    # moculal parameters are taken from CDMS database
    def mole_info():

        mole_info = {}

        connection = sqlite3.connect(db_path)
        cursor = connection.cursor()
        
        query = """
        SELECT * FROM Transitions
        WHERE T_Name = ? 
        AND T_Frequency BETWEEN ? AND ?;
        """
        cursor.execute(query, (name, freq_l, freq_u))
        
        columns = [description[0] for description in cursor.description]
        
        results = [dict(zip(columns, row)) for row in cursor.fetchall()]
        for column in columns:
            mole_info[column] = [entry[column] for entry in results]


        query_pf = "SELECT * FROM Partitionfunctions WHERE PF_Name = ?"
        cursor.execute(query_pf, (name,))
        row = cursor.fetchone()

        columns = [desc[0] for desc in cursor.description]

        if not row:
            raise ValueError(f"No data found for molecule: {name}")

        pf_data = {}
        for key, value in zip(columns, row):
            if key.startswith("PF_") and re.match(r"PF_\d+_\d+", key):
                temp_str = key.split("_", 1)[1].replace('_', '.')
                try:
                    T = float(temp_str)
                    pf_data[T] = value
                except ValueError:
                    continue 

        cursor.close()
        connection.close()

        return mole_info, pf_data


    mole_info, pf_data = mole_info()

    # -------- partation function ------------
    def cal_q(T, pf_data):

        temps = np.array(sorted(pf_data.keys()))
        Qs = np.array([pf_data[T] for T in temps])

        f_interp = interp1d(temps, Qs, kind='linear', bounds_error=True)

        Q = f_interp(T)

        return Q
    

    # -------- beam filling factor ------------
    def apply_beam(source_size, beam):
        
        filling_factor = source_size**2 / (source_size**2+beam**2)

        return filling_factor
    

    # -------- LTE Model ------------
    

    # -------- molecular parameters ------------
    pf_1 = cal_q(p[0][1], pf_data)  
    pf_2 = cal_q(p[1][1], pf_data)      
    mole_info = mole_info
    gu = np.array(mole_info['T_UpperStateDegeneracy'])
    Eu = np.array(mole_info['T_EnergyLower'])  # cm -1
    Aul = np.array(mole_info['T_EinsteinA']) 
    v0 = np.array(mole_info['T_Frequency'])*1e6  # Hz


    T_sim = []

    freq_grid = np.arange(freq_l*1e6, freq_u*1e6, step*1e6)
    tau_l = np.zeros_like(freq_grid)
    tau_l_1 = np.zeros_like(freq_grid)
    tau_l_2 = np.zeros_like(freq_grid)

    # -------- LTE Model ------------
    for i in range(len(v0)):
        
        sigma = ((p[0][3]/ckm)*v0[i]) / (2*np.sqrt(2*np.log(2)))
        v_obs = v0[i] * (1 - p[0][4]/ckm)
        phijk = (1/(np.sqrt(2*np.pi)*sigma)) * np.exp(-(freq_grid - v_obs)**2 / (2*sigma**2))
        tau_0 = ccm**2 / (8*np.pi*freq_grid**2)
        tau_1 = Aul[i]*p[0][2]*((gu[i]*np.exp(-Eu[i]/(kcm*p[0][1]))) / pf_1)
        tau_2 = 1 - np.exp(-h_csg*freq_grid/(k_B*p[0][1]))
        tau = tau_0 * tau_1 * tau_2 * phijk
        tau_l_1 += tau

    for i in range(len(v0)):
        
        sigma = ((p[1][3]/ckm)*v0[i]) / (2*np.sqrt(2*np.log(2)))
        v_obs = v0[i] * (1 - p[1][4]/ckm)
        phijk = (1/(np.sqrt(2*np.pi)*sigma)) * np.exp(-(freq_grid - v_obs)**2 / (2*sigma**2))
        tau_0 = ccm**2 / (8*np.pi*freq_grid**2)
        tau_1 = Aul[i]*p[1][2]*((gu[i]*np.exp(-Eu[i]/(kcm*p[1][1]))) / pf_2)
        tau_2 = 1 - np.exp(-h_csg*freq_grid/(k_B*p[1][1]))
        tau = tau_0 * tau_1 * tau_2 * phijk
        tau_l_2 += tau

    tau_l = tau_l_1 + tau_l_2
    # -------------- 1D two layers radiative transfer eqution ------------------
    filling_factor_1 = apply_beam(p[0][0], beam)

    J_T_1 = (h_csg*freq_grid/k_B) / (np.exp(h_csg*freq_grid/(k_B*p[0][1])) - 1)
    J_T_2 = (h_csg*freq_grid/k_B) / (np.exp(h_csg*freq_grid/(k_B*p[1][1])) - 1)
    J_bg = (h_csg*freq_grid/k_B) / (np.exp(h_csg*freq_grid/(k_B*2.73)) - 1)

    T_sim = filling_factor_1 * ((J_T_1 - J_T_2)*(1 - np.exp(-tau_l)) * np.exp(-p[0][5]) + (J_T_2 - J_bg) * (1 - np.exp(-tau_l)) * np.exp(-(p[0][5]+p[1][5])))
    freq_MHz = freq_grid * 1e-6

    if Save:
        with open(save_file, "w") as f:
            for i, j in zip(freq_MHz, T_sim):
                f.write(f"{i} \t {j}\n")


    return freq_MHz, T_sim

In [ ]:
file_path_37 = "/Users/shangguanlingluo/Library/CloudStorage/OneDrive-mails.ucas.ac.cn/astrophysics/Fitting/13079/CH3NC_peak.tsv"
x_data_37 = []
y_data_37 = []
column1_data_37 = []
column2_data_37 = []

with open(file_path_37, 'r') as file_37:
    lines_37 = file_37.readlines()
    for line_37 in lines_37[11:]:
        columns_37 = line_37.split()
        column1_data_37.append(columns_37[0])
        column2_data_37.append(columns_37[1])

for x, y in zip(column1_data_37, column2_data_37):
    temp_x = float(x)
    temp_y = float(y)
    x_data_37.append(temp_x)
    y_data_37.append(temp_y)

x_data_37 = np.array(x_data_37)
y_data_37 = np.array(y_data_37)


freq_tau, T_sim = gauss_fit(p=[[3, 264, 8.1e16, 3.2, -41.1, 0.92],
                               [3, 122, 5.6e15, 2.4, -38.2, 0.23]], 
                               name='CH3CN;v=0;#1', 
                               freq_l=219367, freq_u=221239, step=0.48, 
                               db_path='/Users/shangguanlingluo/.xclass/db/cdms_lite__official.db', 
                               beam=0.37, 
                               Save=True,
                               save_file='/Users/shangguanlingluo/Library/CloudStorage/OneDrive-mails.ucas.ac.cn/astrophysics/Fitting/13079/fitting/CH3CN_CH3CN_PEAK.txt')

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x = x_data_37, y = y_data_37, mode='lines', line=dict(color='grey', width=1.8), name='Observations', line_shape='hv')) 
fig1.add_trace(go.Scatter(x = freq_tau, y = T_sim, line=dict(color='red', width=1.8), name='Models', line_shape='hv'))
fig1.update_layout(width=2000, height=900)
fig1.show()